In [0]:
%sql
CREATE VOLUME IF NOT EXISTS harshadatabricksdebt.default.machine_stream;

In [0]:
from datetime import datetime
import json
import random

input_path = "/Volumes/harshadatabricksdebt/default/machine_stream"

machines = ["M101", "M102", "M103"]

readings = []

for machine in machines:
    reading = {
        "machine_id": machine,
        "event_time": datetime.now().isoformat(),
        "temperature": round(random.uniform(65, 95), 2),
        "pressure": round(random.uniform(28, 35), 2),
        "vibration": round(random.uniform(1, 6), 2),
        "power_consumption": round(random.uniform(400, 550), 2)
    }

    readings.append(reading)

file_path = f"{input_path}/batch_001.json"

dbutils.fs.put(
    file_path,
    json.dumps(readings),
    overwrite=True
)

print(f"Created: {file_path}")
print(f"Records generated: {len(readings)}")

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType
)

schema = StructType([
    StructField("machine_id", StringType(), True),
    StructField("event_time", StringType(), True),
    StructField("temperature", DoubleType(), True),
    StructField("pressure", DoubleType(), True),
    StructField("vibration", DoubleType(), True),
    StructField("power_consumption", DoubleType(), True)
])

machine_stream_df = (
    spark.readStream
    .schema(schema)
    .json("/Volumes/harshadatabricksdebt/default/machine_stream")
)
checkpoint_path = "/Volumes/harshadatabricksdebt/default/machine_stream/checkpoint"

display(
    machine_stream_df,
    checkpointLocation=checkpoint_path
)


In [0]:
bronze_checkpoint = "/Volumes/harshadatabricksdebt/default/machine_stream/bronze_checkpoint"

bronze_query = (
    machine_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", bronze_checkpoint)
    .toTable("harshadatabricksdebt.default.bronze_machine_readings")
)

print("Bronze ingestion completed")

In [0]:
%sql
SELECT *
FROM harshadatabricksdebt.default.bronze_machine_readings
ORDER BY event_time;